In [ ]:
"""
life_coach_file_search.py
==========================
Life Coach Agent with:
  - FileSearchTool (personal goals & journal from vector store)
  - WebSearchTool (evidence-based advice from the web)
  - Session memory (multi-turn conversation via to_input_list)
  - Streamlit chat UI

Setup (run once before starting):
    python setup_vector_store.py

Run:
    streamlit run life_coach_file_search.py
"""

import os
import streamlit as st
from dotenv import load_dotenv
from agents import Agent, Runner, FileSearchTool, WebSearchTool

# ============================================================
# 1. Load environment & page config
# ============================================================
load_dotenv()

st.set_page_config(
    page_title="Life Coach Agent",
    page_icon="🌱",
    layout="centered",
)


In [ ]:
# ============================================================
# 2. Validate vector store ID
# ============================================================
VECTOR_STORE_ID = os.getenv("VECTOR_STORE_ID", "")

if not VECTOR_STORE_ID:
    st.error(
        "⚠️ VECTOR_STORE_ID가 설정되지 않았습니다.\n\n"
        "먼저 `python setup_vector_store.py`를 실행하여 "
        "벡터 스토어를 생성하세요."
    )
    st.stop()

# ============================================================
# 3. Agent definition
# ============================================================
@st.cache_resource
def create_agent():
    return Agent(
        name="Life Coach",
        instructions="""You are a warm, encouraging, and knowledgeable Life Coach who has access to the user's personal goals document and journal entries.

YOUR TOOLS:
1. **File Search** — Search the user's personal goals, journal entries, and progress records.
   Use this FIRST whenever the user asks about their goals, progress, habits, or past entries.
2. **Web Search** — Search the internet for evidence-based tips, research, and advice.
   Use this to supplement your advice with up-to-date information.

WORKFLOW (follow this order):
1. When the user asks a question, FIRST search their goals document to understand their personal context.
2. THEN search the web for relevant, evidence-based advice tailored to their situation.
3. Combine both sources to give personalized, actionable recommendations.

BEHAVIOR:
- Always reference specific details from the user's goals document (dates, numbers, progress).
- Compare current progress to stated goals and acknowledge improvements.
- Give concrete, step-by-step actionable advice (not vague platitudes).
- Celebrate the user's progress — even small wins matter.
- If the user is behind on a goal, be encouraging, not judgmental.
- Track patterns across journal entries (e.g., "2월에 운동이 주 1회에서 2회로 늘었네요!").
- End responses with an encouraging message or a follow-up question.

TOPICS YOU HELP WITH:
- Morning routines, sleep, waking up early
- Exercise and fitness goals
- Habit formation and habit stacking
- Reading, learning, self-development
- Mindfulness, meditation, mental health
- Productivity, screen time, SNS management
- Career development and skill building
- Goal tracking and accountability

Respond in Korean.""",
        tools=[
            FileSearchTool(
                max_num_results=5,
                vector_store_ids=[VECTOR_STORE_ID],
            ),
            WebSearchTool(),
        ],
        model="gpt-4o-mini",
    )


agent = create_agent()

In [ ]:
# ============================================================
# 4. Session state — conversation memory
# ============================================================
if "input_history" not in st.session_state:
    st.session_state.input_history = []

if "display_messages" not in st.session_state:
    st.session_state.display_messages = []

In [ ]:
# ============================================================
# 5. UI — Header & info
# ============================================================
st.title("🌱 Life Coach Agent")
st.caption(
    "개인 목표 문서를 기반으로 맞춤형 조언을 제공합니다. "
    "목표 진행 상황, 습관 형성, 자기 개발에 대해 물어보세요!"
)

# Sidebar — show current tools info
with st.sidebar:
    st.header("🔧 에이전트 도구")
    st.markdown(
        f"""
        **📂 파일 검색 (File Search)**
        - Vector Store: `{VECTOR_STORE_ID[:20]}...`
        - 개인 목표, 일기, 진행 상황 검색

        **🔍 웹 검색 (Web Search)**
        - 최신 자기 개발 팁 검색
        - 과학적 근거 기반 조언 제공
        """
    )
    st.divider()
    st.markdown("**💬 대화 예시:**")
    example_prompts = [
        "내 운동 목표 달성은 잘 되어가고 있어?",
        "내가 읽은 책에서 배운 점을 알려줘",
        "명상 습관을 더 잘 유지하려면 어떻게 해야 해?",
        "SNS 사용 줄이는 팁 알려줘",
        "이번 달 전체 목표 달성률은 어때?",
    ]
    for prompt in example_prompts:
        st.caption(f"• {prompt}")

# ============================================================
# 6. UI — Render chat history
# ============================================================
for msg in st.session_state.display_messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ============================================================
# 7. UI — Chat input + Agent execution
# ============================================================
user_input = st.chat_input("메시지를 입력하세요...")

if user_input:
    # --- Display user message ---
    st.session_state.display_messages.append(
        {"role": "user", "content": user_input}
    )
    with st.chat_message("user"):
        st.markdown(user_input)

    # --- Build input for the agent ---
    agent_input = st.session_state.input_history + [
        {"role": "user", "content": user_input}
    ]

    # --- Run agent with spinner ---
    with st.chat_message("assistant"):
        with st.spinner("목표 문서를 검색하고, 맞춤형 조언을 준비하고 있어요... 📂🔍"):
            result = Runner.run_sync(
                agent,
                input=agent_input,
            )
        st.markdown(result.final_output)

    # --- Update session memory ---
    st.session_state.input_history = result.to_input_list()

    # Save for display
    st.session_state.display_messages.append(
        {"role": "assistant", "content": result.final_output}
    )
